In [6]:
# trade_charts_daily.py
# -*- coding: utf-8 -*-
"""
Her ticker için son 1 ayın GÜNLÜK fiyatlarını yfinance'tan çeker,
ALIŞ/SATIŞ noktalarını grafiğe koyar ve sadece İŞLEM FİYATINI (adet yok) yazdırır.
PDF ÜRETMEZ. Her ticker için tek bir PNG dosyası kaydeder.

Kurulum:
    pip install pandas numpy matplotlib yfinance
Çalıştırma:
    python trade_charts_daily.py
"""

import os
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- Gömülü işlemler (senin son paylaştığın tablo) ----
# Kolonlar: Ticker, Tarih, Tutar($), Ücret, Fiyat, İşlem
TRADES = [
    ("SPGI",  "8.10.2025",   480.61,     0.96,  480.61, "ALIŞ"),
    ("AMT",   "13.10.2025",  369.30,     0.73,  184.65, "ALIŞ"),
    ("BSX",   "13.10.2025",  284.73,     0.56,   94.91, "ALIŞ"),
    ("HD",    "13.10.2025",  378.70,     0.76,  378.70, "ALIŞ"),
    ("PG",    "13.10.2025",  297.90,     0.59,  148.95, "ALIŞ"),
    ("DAL",   "15.10.2025",  122.96,     0.50,   61.48, "ALIŞ"),
    ("DAL",   "20.10.2025",  122.34,     0.50,   61.17, "SATIŞ"),
    ("NVDA",  "15.10.2025",  180.00,     1.50,  179.57, "ALIŞ"),
    ("BRK.B", "15.10.2025",  125.00,     1.50,  492.89, "ALIŞ"),
    ("AAPL",  "16.10.2025",  150.00,     1.50,  247.39, "ALIŞ"),
    ("AAPL",  "20.10.2025",  156.8090869,1.50,  258.62, "SATIŞ"),
    ("STX",   "22.10.2025",  213.13,     1.50,  213.13, "ALIŞ"),
    ("STX",   "23.10.2025",  225.03,     1.50,  225.03, "SATIŞ"),
    ("META",  "24.10.2025",  366.48,     1.50,  732.95, "ALIŞ"),
    ("VOO",   "24.10.2025",  311.29,     1.50,  622.57, "ALIŞ"),
]

# Yahoo Finance sembol eşleştirmesi (örn. BRK.B -> BRK-B)
YF_MAPPING = {
    "BRK.B": "BRK-B",
}

def parse_date_tr(x: str) -> datetime:
    for fmt in ("%d.%m.%Y", "%d.%m.%y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(str(x), fmt)
        except Exception:
            pass
    # yine de parse edilemezse NaT dönebilir
    return pd.to_datetime(x, dayfirst=True, errors="coerce")

def prepare_trades(trades):
    df = pd.DataFrame(trades, columns=["Ticker","Tarih","Amount","Fee","TradePrice","Side"])
    df["Date"] = df["Tarih"].apply(parse_date_tr)
    # sadece ihtiyacımız olan kolonlar
    df = df[["Ticker","Date","TradePrice","Side"]].sort_values(["Ticker","Date"]).reset_index(drop=True)
    return df

def fetch_daily_prices(ticker: str, start: datetime, end: datetime) -> pd.Series:
    """yfinance'tan günlük kapanışlar. Başarısız olursa boş döner."""
    try:
        import yfinance as yf
        yf_ticker = YF_MAPPING.get(ticker, ticker)
        data = yf.download(yf_ticker, start=start, end=end, interval="1d", progress=False, threads=True)
        if data is None or len(data) == 0:
            return pd.Series(dtype=float)
        close = data["Close"].copy()
        close.name = "Close"
        return close
    except Exception:
        return pd.Series(dtype=float)

def plot_ticker_daily(ax, close: pd.Series, trades_tkr: pd.DataFrame, ticker: str):
    # Fiyat çizgisi
    ax.plot(close.index, close.values, linewidth=1.5)
    ax.set_title(f"{ticker} — Son 1 Ay Günlük Fiyat")
    ax.set_xlabel("Tarih"); ax.set_ylabel("Fiyat")
    ax.grid(True, linestyle="--", alpha=0.3)

    # ALIŞ / SATIŞ noktaları (sadece fiyat etiketi)
    buys  = trades_tkr[trades_tkr["Side"].str.upper().str.startswith("ALI")]
    sells = trades_tkr[~trades_tkr["Side"].str.upper().str.startswith("ALI")]

    # ALIŞ: ▲ ve fiyat etiketi
    if not buys.empty:
        ax.scatter(buys["Date"], buys["TradePrice"], marker="^", s=70, label="ALIŞ")
        for _, r in buys.iterrows():
            ax.annotate(f"{r['TradePrice']:.2f}", (r["Date"], r["TradePrice"]),
                        textcoords="offset points", xytext=(0,8), ha="center", fontsize=8)

    # SATIŞ: ▼ ve fiyat etiketi
    if not sells.empty:
        ax.scatter(sells["Date"], sells["TradePrice"], marker="v", s=70, label="SATIŞ")
        for _, r in sells.iterrows():
            ax.annotate(f"{r['TradePrice']:.2f}", (r["Date"], r["TradePrice"]),
                        textcoords="offset points", xytext=(0,-12), ha="center", fontsize=8)

    if (not buys.empty) or (not sells.empty):
        ax.legend(loc="best")

def main():
    os.makedirs("charts", exist_ok=True)

    trades = prepare_trades(TRADES)
    tickers = trades["Ticker"].unique()

    # Pencere: son işlemin tarihine göre 1 ay (yoksa bugüne göre)
    max_trade_dt = trades["Date"].max()
    base_end = max_trade_dt if pd.notna(max_trade_dt) else datetime.utcnow()
    start = (base_end - timedelta(days=31)).replace(hour=0, minute=0, second=0, microsecond=0)
    end   = (base_end + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)

    for tkr in tickers:
        tdf = trades[trades["Ticker"] == tkr].copy()

        # Günlük kapanışları çek
        close = fetch_daily_prices(tkr, start, end)

        # Eğer veri gelmezse, en azından trade noktalarını çizmek için bir fallback çizgi yapalım:
        # (kullanıcı "gün gün verileri çek" dedi, o yüzden fallback kullanmamak daha doğru;
        #  yine de boş kalmasın diye basit trade-bazlı çizgi ekliyoruz)
        if close.empty:
            # Trade günleri için seri, diğer günler NaN; ffill ile basit çizgi
            idx = pd.date_range(start=start, end=end, freq="D")
            s = pd.Series(index=pd.to_datetime(tdf["Date"]), data=tdf["TradePrice"].values).sort_index()
            s = s[~s.index.duplicated(keep="last")]
            close = s.reindex(idx).ffill()
            close.name = "Close"

        # Çiz ve kaydet
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_ticker_daily(ax, close, tdf, tkr)
        plt.tight_layout()
        out_png = os.path.join("charts", f"{tkr}.png")
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

    print("Tamamdır. Grafikler charts/ klasörüne kaydedildi (her ticker için 1 PNG).")

if __name__ == "__main__":
    main()


Tamamdır. Grafikler charts/ klasörüne kaydedildi (her ticker için 1 PNG).
